# PDF → CSV — working copy

**Copy this file before using it.** Work in the copy, one per job, so a
notebook you have broken is never the only one you have.

This is the workbench. The web interface gives you a button; this gives you
the intermediate values — use it when a document fails validation, when you
are working out the rules for an unfamiliar bank format, or when you need to
see *why* a column came out the way it did.

### The one rule

**No logic in this notebook.** It calls `run()` and `export()` and nothing
else. Cells get run out of order, edited and skipped — including, sooner or
later, a validation cell. Since validation lives *inside* `export()`, there is
no way to produce a CSV here that skipped it, however the cells are run.

If you find yourself writing extraction logic in a cell below, it belongs in
`pdf2csv/core/` with a test.

In [ ]:
# Cell 1 — setup
import sys

sys.path.insert(0, "../src")

from pdf2csv import run
from pdf2csv.core.export import export
from pdf2csv.logging_setup import setup_logging

setup_logging(level="INFO")

In [ ]:
# Cell 2 — choose a document
#
# Set the path directly, or use the file picker if ipyfilechooser is
# installed (pip install ipyfilechooser).

PDF = "../tests/fixtures/pdfs/statement_ruled_2page.pdf"

try:
    from ipyfilechooser import FileChooser

    chooser = FileChooser(".", filter_pattern="*.pdf")
    display(chooser)  # noqa: F821 — provided by IPython
except ImportError:
    chooser = None
    print(f"Using: {PDF}")

In [ ]:
# Cell 3 — run and export
#
# export() is the gate. It prints every failed check and writes the
# .validation.json sidecar next to the CSV. There is no way to write a CSV
# from here that bypasses it.

source = (chooser.selected if chooser and chooser.selected else PDF)

result = run(source)
export(result, "output.csv")

result.dataframe.head(20)

---
## Looking closer

Everything below is optional and read-only. Nothing here changes the output.

In [ ]:
# How each page was read, and which strategy produced each table.
# The first thing to check when a document comes out wrong: a page routed to
# OCR that should have been read as text, or the reverse.

print(f"{result.meta.source_name}  —  {result.meta.duration_seconds:.1f}s")
for number, kind in enumerate(result.meta.page_kinds, start=1):
    print(f"  page {number}: {kind.value}")

print()
for table in result.tables:
    print(
        f"  page {table.page_number}: {table.n_rows}x{table.n_cols}"
        f" via {table.extractor}"
        f" (min confidence {table.min_confidence():.0%})"
    )

In [ ]:
# Every check, with the detail and the suggested action.

for check in result.report.checks:
    mark = "PASS" if check.passed else check.severity.value.upper()
    print(f"[{mark:7s}] {check.title}")
    print(f"          {check.detail}")
    if check.hint:
        print(f"          -> {check.hint}")
    print()

In [ ]:
# The rows the report flagged — go here first when a check fails.

flags = result.report.flags
if not flags:
    print("No flagged cells.")
else:
    for flag in flags:
        print(f"row {flag.row + 1:>4}  {flag.column:<20} {flag.reason}")
        if flag.value:
            print(f"{'':>6}  the PDF showed: {flag.value!r}")

    rows = sorted({f.row for f in flags})
    result.dataframe.iloc[rows]

In [ ]:
# Why each column was given the type it was.
# Useful when an amount column came out as text, or a reference column came
# out as a number and lost its leading zeros.

print(result.dataframe.dtypes.to_string())

In [ ]:
# Trying a different document profile, without touching any code.
# Profiles live in src/pdf2csv/profiles/ — see example_bank_statement.yaml.

# alternative = run(source, profile="example_bank_statement")
# alternative.dataframe.head(20)